# Chapter 23 — Did the Context Help?

## Question

**If a compiler produced a legal, budget-compliant bundle, did supplying that bundle actually improve what the reader did?**

This notebook separates bundle-construction quality from behavioural utility. It transcribes one frozen construction result, verifies its arithmetic, and builds the matched-intervention machinery the behavioural question requires — with behavioural rows left explicitly NOT_RUN.

## Frozen book result — construction (transcribed, not produced)

> FROZEN BOOK RESULT — TRANSCRIBED FROM THE MANUSCRIPT. NOT GENERATED BY THIS NOTEBOOK.

Experiment `compiler-v1`, run `run-001`, implementation commit `ec642dc`, artifact validated clean: 14 fixtures × 3 budgets × 6 strategies = 252 compilations. Staged: correct status on all 42 fixture-budget combinations, required-item recall 1.0 on all 34 feasible combinations, zero scope/freshness/authority/floor/dependency violations, explicit failures with exact reason codes on every infeasible combination.

In [ ]:
FROZEN_BOOK_RESULT = {
    'experiment': 'compiler-v1',
    'run': 'run-001',
    'commit': 'ec642dc',
    'compilations': 252,
    'combinations': 42,
    'feasible': 34,
    'table': {
        'dump/truncate':     {'status': (34, 42), 'dependency': 1, 'group': 0, 'floor': 3, 'distractor': 18, 'scope': 5, 'freshness': 4},
        'top-k':             {'status': (34, 42), 'dependency': 1, 'group': 1, 'floor': 3, 'distractor': 18, 'scope': 6, 'freshness': 6},
        'weighted':          {'status': (34, 42), 'dependency': 1, 'group': 0, 'floor': 3, 'distractor': 17, 'scope': 6, 'freshness': 6},
        'hard-gated greedy': {'status': (34, 42), 'dependency': 1, 'group': 1, 'floor': 0, 'distractor': 9, 'scope': 0, 'freshness': 0},
        'staged compiler':   {'status': (42, 42), 'dependency': 0, 'group': 0, 'floor': 0, 'distractor': 5, 'scope': 0, 'freshness': 0},
        'oracle':            {'status': (42, 42), 'dependency': 0, 'group': 0, 'floor': 0, 'distractor': 0, 'scope': 0, 'freshness': 0},
    },
    'staged_distractors': 5,
    'staged_harmful': 2,
    'oracle_gap_tokens': (-210, 676, 51),
    'must_recall_greedy': 0.667,
    'must_recall_staged': 1.0,
}
r = FROZEN_BOOK_RESULT
assert r['combinations'] == 42 and r['feasible'] == 34
assert 14 * 3 * 6 == r['compilations']
assert r['table']['staged compiler']['status'] == (42, 42)
print('frozen construction record loaded; notebook generated none of it.')

## Baseline — derived checks on the frozen table

In [ ]:
t = r['table']
for name, row in t.items():
    illegal = row['scope'] + row['freshness'] + row['floor'] + row['dependency'] + row['group']
    print(f"{name:17s} status {row['status'][0]}/{row['status'][1]}  illegal-kind={illegal:3d}  distractors={row['distractor']}")
staged, greedy = t['staged compiler'], t['hard-gated greedy']
assert staged['distractor'] == 5 and staged['scope'] + staged['freshness'] == 0
assert greedy['distractor'] == 9 and r['must_recall_staged'] - r['must_recall_greedy'] > 0.3
print('Zero illegal admissions does not imply zero distractors (5 staged), hence cannot imply utility.')

## Intervention — influence is not utility (synthetic illustration)

> SYNTHETIC illustration below: Condition A action = HOLD score 1; Condition B action = RELEASE score 0. Behaviour changed (influence) while the outcome worsened (negative utility). No frozen row is claimed.

In [ ]:
A = {'action': 'HOLD', 'score': 1}
B = {'action': 'RELEASE', 'score': 0}
influence = A['action'] != B['action']
utility = B['score'] - A['score']
print(f'influence (action changed): {influence}; utility (score delta): {utility}')
assert influence is True and utility < 0
print('Movement is evidence of influence, never evidence of success.')

## Attribution ladder and matched-intervention contract

In [ ]:
LADDER = ['PRESENT', 'MENTIONED', 'CONSISTENT', 'REMOVAL_SENSITIVE', 'RESTORATION_CONFIRMED']
print('attribution ladder:', ' -> '.join(LADDER))
print('Presence and citation never count as causal use.')

PLAN = {'fixed': ['task', 'reader', 'prompt', 'decoding', 'grader'], 'varied': ['ContextBundle']}
print(f"matched plan: fixed={PLAN['fixed']}; varied={PLAN['varied']}")
assert PLAN['varied'] == ['ContextBundle']

CONDITIONS = ['no-context', 'dump', 'top-k', 'weighted', 'hard-gated greedy', 'staged', 'oracle']
BEHAVIOURAL_ROWS = {c: 'NOT_RUN' for c in CONDITIONS}
print('behavioural rows:', BEHAVIOURAL_ROWS)
assert all(v == 'NOT_RUN' for v in BEHAVIOURAL_ROWS.values())

## Remove / restore with integrity proofs, outcomes pending

In [ ]:
import hashlib

parent = b'staged-bundle-bytes'
minus_one = b'staged-bundle-bytes-minus-decisive'
restored = parent  # byte-identical restoration path
token_matched = b'volume-without-evidence'
wrong_context = b'plausible-falsehood'

def digest(b):
    return hashlib.sha256(b).hexdigest()[:12]

print(f"parent={digest(parent)} restored={digest(restored)} identical={digest(parent) == digest(restored)}")
assert digest(parent) == digest(restored), 'restore integrity proven'
print('Causal attribution not proven: outcomes below are NOT_RUN, not zero.')
OUTCOMES = {'parent': 'NOT_RUN', 'minus': 'NOT_RUN', 'restored': 'NOT_RUN',
            'token_matched': 'NOT_RUN', 'wrong_context': 'NOT_RUN', 'no_context': 'NOT_RUN'}
assert set(OUTCOMES) == {'parent', 'minus', 'restored', 'token_matched', 'wrong_context', 'no_context'}
print('No-context anchors every task: necessity, interference, and leakage calibration — pending runs.')

## Observation — two ledgers, never one score

In [ ]:
construction = {'illegal_admissions': 0, 'distractors': 5, 'status': '42/42'}
behavioural = {'success': 'NOT_RUN', 'harm': 'NOT_RUN', 'action_change': 'NOT_RUN'}
print(f"CONSTRUCTION EVIDENCE: {construction}")
print(f"BEHAVIOURAL EVIDENCE:  {behavioural}")
assert construction['illegal_admissions'] == 0 and construction['distractors'] == 5
print('A legal bundle is not necessarily a useful bundle. The second ledger is empty until run.')

## Try it

1. Recompute illegal-kind totals with distractor columns excluded: gates earn their keep against ungated packers on violations alone.
2. Change the synthetic B score to 1 and confirm influence stays True while utility goes to zero — movement without benefit.
3. Verify the frozen dict against the chapter table cell by cell; any mismatch is a transcription bug to report, not a result.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(FROZEN_BOOK_RESULT['table']['staged compiler'])

## What this demonstrates

- Bundle correctness and behavioural usefulness are different evidence claims, held in separate ledgers.
- Influence and benefit are different: changed action with worsened score is influence without utility.
- Matched interventions (bundle-only variation, remove/restore with digests, no-context anchors) are required to move from properties to claims.
- Remove/restore integrity is provable structurally; causal attribution stays pending without runs.

## What this does not demonstrate

- That staged beats greedy behaviourally, or that legal means useful.
- That distractor labels imply behavioural harm, or that the oracle is the behavioural optimum.
- That mention or citation proves causal use, or that one reader generalises to another.
- Any real-world prevalence from synthetic fixtures.

## Connection to the chapter

Construction is closed. Behaviour is still open:

> The final question is no longer what mechanisms we can imagine, but which mechanisms the evidence actually earned.

That is Chapter 24.